# ISOM 839 · Session 10 Bonus — Is Deep Learning Still Relevant? 🏁 The Great Tabular Race

**Prescriptive Analytics · Suffolk University · Prof. Hasan Arslan**

A new kind of model can read a whole spreadsheet of customers and predict a new one **without being trained on it**. It's called a *tabular foundation model* (TabICL, TabPFN). So do we still need deep learning?

You'll find out by racing four models on the same customers, and then by watching a neural network learn the **price tags** from tonight's session.

| Part | What you do | Time |
|---|---|---|
| **0 · Setup** | Run two cells | 2 min |
| **1 · Meet the customers** | 25,000 subscribers, about 1 in 4 leave | 3 min |
| **2 · The race** | Four models, 300 customers, one winner | 5 min |
| **3 · More data** | Who wins when the data grows? | 5 min |
| **4 · Back to tonight** | A neural network learns a shop's price tags | 8 min |

| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter** |
| 🤔 **Bet first** | guess out loud before you run |
| ✅ **What you should see** | if your output matches, move on |
| ✏️ **Your turn** | change something and re-run |

💡 **Tip:** *Runtime → Change runtime type → T4 GPU* makes the foundation model much faster. It works without one too, just slower.

---
## Part 0 — Setup

In [ ]:
# @title ▶️ Run me: install the foundation model (about 1 minute)
!pip install -q tabicl
print('✅ Installed')

In [ ]:
# @title ▶️ Run me: load the tools
import time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

try:
    import torch
    from tabicl import TabICLClassifier
    GPU = torch.cuda.is_available()
    HAVE_TABICL = True
except Exception as e:
    GPU, HAVE_TABICL = False, False
    print('⚠️ TabICL could not load:', e, '\n   Everything else still works. Ask for help, or re-run the install cell.')

def card(title, rows, color='#0a66c2'):
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:720px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready!', '| GPU:', 'yes 🚀' if GPU else 'no (fine, just slower)', '| TabICL:', 'ready' if HAVE_TABICL else 'missing')

**✅ What you should see:** `✅ Ready!` and `TabICL: ready`. GPU yes or no are both fine.

---
## Part 1 — Meet the customers

A streaming service with 25,000 subscribers. For each one we know 7 things, and whether they **churned** (left) last quarter.

Behind the scenes, a **hidden rule** decides who leaves: new customers are fragile, month-to-month plans are risky, every support call after the first hurts, and expensive plans plus problems is a bad mix. The models never see this rule. They have to discover it from examples, just like real life.

In [ ]:
# @title ▶️ Run me: create the customers
def make_customers(n, seed=839):
    rng = np.random.default_rng(seed)
    tenure   = rng.integers(1, 73, n)
    contract = rng.choice(['month-to-month', 'one-year', 'two-year'], n, p=[0.55, 0.25, 0.20])
    charge   = np.round(rng.normal(70, 25, n).clip(20, 140), 2)
    calls    = rng.poisson(1.2, n)
    autopay  = rng.random(n) < 0.45
    usage    = np.round(rng.gamma(2.0, 6.0, n), 1)
    age      = rng.integers(18, 80, n)
    # 🙈 the hidden rule (no peeking until the end!)
    z = (-1.2 + 1.4 * (contract == 'month-to-month') - 0.8 * (contract == 'two-year')
         - 1.6 * (1 - np.exp(-tenure / 12))
         + 0.025 * (charge - 70) * (contract == 'month-to-month')
         + 0.8 * np.maximum(calls - 1, 0)
         - 0.6 * autopay
         + 0.9 * ((usage < 3) & (tenure < 12))
         + 0.0012 * (age - 45) ** 2
         + 1.2 * ((charge > 95) & (calls >= 2))
         - 0.3)
    p_true = 1 / (1 + np.exp(-z))
    churn = (rng.random(n) < p_true).astype(int)
    df = pd.DataFrame(dict(tenure_months=tenure, contract=contract, monthly_charge=charge, support_calls=calls,
                           autopay=autopay.astype(int), data_gb=usage, age=age, churned=churn))
    return df, p_true

customers, p_true = make_customers(25_000)
X = pd.get_dummies(customers.drop(columns='churned'), columns=['contract'], dtype=float)
y = customers['churned'].values

# the last 5,000 customers are the exam: no model ever trains on them
X_exam, y_exam = X.iloc[20_000:].values, y[20_000:]
display(customers.head())
print(f'Churn rate: {y.mean():.1%}')

**✅ What you should see:** a table of 5 customers and `Churn rate: 23.4%`.

### How we score: AUC, in one sentence
> Pick one customer who left and one who stayed. **AUC is how often the model says the leaver was riskier.**

0.5 = a coin flip. 1.0 = perfect. A *perfect crystal ball* that knew the hidden rule still wouldn't reach 1.0, because people are random. We'll reveal its score at the end of Part 3.

---
## Part 2 — The race: four models, 300 customers

| Contestant | How it learns | Analogy |
|---|---|---|
| **Logistic regression** | one straight-line formula | a simple checklist |
| **XGBoost** | hundreds of small decision trees | a committee of rule-of-thumb experts |
| **Neural network** | layers of formulas tuned on *your* data | a bright intern who needs lots of practice |
| **TabICL** (foundation model) | **pretrained on millions of made-up tables.** It reads your 300 rows like a prompt and answers in one pass. | a veteran who has seen a million businesses and glances at yours |

**🤔 Bet first:** with only **300** customers to learn from, who wins?

In [ ]:
# @title ▶️ Run me: the four contestants
def contestants(n_train=300):
    c = {
        'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        'XGBoost':             XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, verbosity=0),
        'Neural network':      make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-2, max_iter=800, random_state=0)),
    }
    if HAVE_TABICL and (GPU or n_train <= 3_000):          # without a GPU we skip the biggest size to save time
        c['TabICL (foundation model)'] = TabICLClassifier()
    return c

def race(n_train, show=True):
    Xtr, ytr = X.iloc[:n_train].values, y[:n_train]
    results = {}
    for name, model in contestants(n_train).items():
        t = time.time()
        model.fit(Xtr, ytr)
        auc = roc_auc_score(y_exam, model.predict_proba(X_exam)[:, 1])
        results[name] = (auc, time.time() - t)
        if show:
            print(f'{name:27s} AUC {auc:.3f}   ({time.time() - t:5.1f} s)')
    return results

r300 = race(300)
names = list(r300)
plt.figure(figsize=(8, 3))
bars = plt.barh(names, [r300[k][0] for k in names], color=['#9aa5b1', '#4c9be8', '#f2a541', '#2bb673'][:len(names)])
plt.xlim(0.5, 0.85); plt.xlabel('AUC on the 5,000-customer exam (higher is better)'); plt.title('The race at 300 customers')
for b, k in zip(bars, names): plt.text(r300[k][0] + 0.003, b.get_y() + 0.25, f'{r300[k][0]:.3f}')
plt.gca().invert_yaxis(); plt.show()

**✅ What you should see:** four bars. With 300 customers, the **neural network trained from scratch is last** (about 0.71): it's a bright intern with too little practice. **XGBoost** lands around 0.73 and the simple **logistic regression** around 0.77. **TabICL** is usually at or near the top, *without ever being trained on this business*. (Exact numbers vary a little by computer.)

Notice the timing column too: TabICL's `fit()` just *stores* the 300 rows. All its thinking happens in `predict`, when it reads them.

### 🕵️ Is the foundation model cheating?
Maybe it memorized churn data somewhere? Test it: give it the **same customers with the churn labels shuffled** (pure nonsense). If it really reads our data, its score should collapse to a coin flip.

In [ ]:
# @title ▶️ Run me: the shuffled-labels test
if HAVE_TABICL:
    rng = np.random.default_rng(1)
    nonsense = rng.permutation(y[:300])                       # same customers, labels scrambled
    m = TabICLClassifier().fit(X.iloc[:300].values, nonsense)
    print(f'Real labels:     AUC {r300["TabICL (foundation model)"][0]:.3f}')
    print(f'Shuffled labels: AUC {roc_auc_score(y_exam, m.predict_proba(X_exam)[:, 1]):.3f}')
else:
    print('TabICL is not loaded; skip this one.')

**✅ What you should see:** the shuffled version drops to about **0.5**, a coin flip. So it isn't remembering an answer. It learned *how to learn from a table* during pretraining, and it really reads **your** rows. That trick is called **in-context learning**: the dataset is the prompt.

---
## Part 3 — More data: does the winner change?

Same race at 100, 300, 1,000, 3,000 and 10,000 customers.

**🤔 Bet first:** which line climbs the most as data grows?

In [ ]:
# @title ▶️ Run me: the race at five sizes (a few minutes; faster on a GPU)
sizes = [100, 300, 1_000, 3_000, 10_000]
curves = {k: [np.nan] * len(sizes) for k in contestants()}
for i, n in enumerate(sizes):
    res = race(n, show=False)
    for k, (auc, secs) in res.items():
        curves[k][i] = auc
    print(f'{n:>6,} customers:  ' + '   '.join(f'{k.split(" (")[0]} {v[0]:.3f}' for k, v in res.items()))

crystal_ball = roc_auc_score(y_exam, p_true[20_000:])
plt.figure(figsize=(8, 4))
colors = dict(zip(curves, ['#9aa5b1', '#4c9be8', '#f2a541', '#2bb673']))
for k, v in curves.items():
    plt.plot(sizes, v, marker='o', label=k, color=colors[k], lw=2.5 if 'TabICL' in k else 1.8)
plt.axhline(crystal_ball, color='black', ls='--', lw=1); plt.text(110, crystal_ball + 0.004, f'perfect crystal ball {crystal_ball:.3f}')
plt.xscale('log'); plt.xticks(sizes, ['100', '300', '1,000', '3,000', '10,000']); plt.minorticks_off(); plt.xlabel('customers to learn from'); plt.ylabel('AUC on the exam'); plt.ylim(0.45, 0.85)
plt.legend(loc='lower right'); plt.title('Who wins depends on how much data you have'); plt.show()

*(No GPU? TabICL sits out the 10,000 round to save time. Its line simply stops at 3,000.)*

**✅ What you should see:** the **perfect crystal ball** line at **0.814**, the best anyone could do. Logistic regression is good early but **stops improving** around 0.785: a straight line can't capture "new *and* barely using it." The **neural network** climbs the most, from last place to about **0.80** at 10,000 customers, next to XGBoost. TabICL starts near the top.

### What the race teaches
| Situation | Smart first choice |
|---|---|
| A small or medium table, need an answer today | **Foundation model** (TabICL, TabPFN). No tuning, strong at small sizes. |
| A medium or large table | **XGBoost** remains a tough, fast benchmark, and foundation models now compete here too. |
| Huge data, images, text, sensors, or a custom goal | **Your own neural network**, or fine-tune a foundation model. |
| You must explain every decision to a regulator | **Logistic regression** or a small tree, then compare to the others. |

And the twist: **TabICL *is* deep learning.** It's a transformer, trained by people who know deep learning very well. Deep learning didn't disappear. It **moved up a floor**: from "train a network on your data" to "use a network someone pretrained, and know when to adapt it."

---
## Part 4 — Back to tonight: a neural network learns the price tags

Tonight's shop had **one** product and **11** price tags (one per shelf level). Now a shop with **two products**, each shelf 0–10, so **121 situations**. They share **one truck** (5 units total, **$5 per delivery**). Same money as tonight: +\$10 a sale, −\$8 a unit, −\$15 per customer turned away, γ = 0.95.

With 121 states we can still solve it exactly. A real store with 1,000 products can't (a 1,042-digit number of states). So let's pretend we can only afford to compute the price tags of **20 situations**, and ask a neural network to **fill in the other 101**.

### Step 4.1 — Solve the two-product shop exactly (our answer key)

In [ ]:
# @title ▶️ Run me: value iteration for the two-product shop
N, GAMMA, FEE, TRUCK = 11, 0.95, 5, 5
demand, p_d = np.arange(4), np.full(4, 0.25)
L = np.zeros((N, N)); g = np.zeros(N)            # L[t, left] = chance of 'left' units tonight if the shelf starts at t
for t in range(N):
    for d, p in zip(demand, p_d):
        sold = min(t, d); L[t, t - sold] += p; g[t] += p * (10 * sold - 15 * (d - sold))
actions = [(a1, a2) for a1 in range(TRUCK + 1) for a2 in range(TRUCK + 1) if a1 + a2 <= TRUCK]

def scores(V):
    # Q[s1, s2, action] = cash today + gamma x average price tag of tomorrow (Bellman, both products at once)
    W = L @ V @ L.T
    Q = np.full((N, N, len(actions)), -np.inf)
    for k, (a1, a2) in enumerate(actions):
        for s1 in range(N - a1):
            s2 = np.arange(N - a2); t1, t2 = s1 + a1, s2 + a2
            Q[s1, s2, k] = g[t1] + g[t2] - 8 * (a1 + a2) - (FEE if a1 + a2 > 0 else 0) + GAMMA * W[t1, t2]
    return Q

V_true = np.zeros((N, N))
for sweep in range(1, 10_000):
    V_new = scores(V_true).max(axis=2)
    if np.abs(V_new - V_true).max() < 1e-6:
        V_true = V_new; break
    V_true = V_new
best_play = scores(V_true).argmax(axis=2)
print(f'{N*N} situations, {len(actions)} possible orders, converged in {sweep} sweeps')
print('Example: shelves (0, 0) -> order', actions[best_play[0, 0]], '   shelves (3, 3) -> order', actions[best_play[3, 3]])

**✅ What you should see:** `121 situations, 21 possible orders, converged in 281 sweeps`. With both shelves empty it orders **(2, 3)**, a full truck split across the products. At (3, 3) it **waits**. That's the "wait, then fill the truck" idea from the class game, now juggling two products.

### Step 4.2 — Learn the price tags from just 20 situations

In [ ]:
# @title ▶️ Run me: a neural network vs a straight line
SAMPLES = 20                                              # ✏️ try 10 or 50 later
grid = np.array([(a, b) for a in range(N) for b in range(N)])
rng = np.random.default_rng(839)
seen = grid[rng.choice(len(grid), SAMPLES, replace=False)]
tags = V_true[seen[:, 0], seen[:, 1]]
mu, sd = tags.mean(), tags.std()

net  = MLPRegressor(hidden_layer_sizes=(32, 32), activation='tanh', solver='lbfgs', max_iter=5000,
                    alpha=1e-4, random_state=0).fit(seen / 10, (tags - mu) / sd)
line = LinearRegression().fit(seen / 10, tags)
V_net  = (net.predict(grid / 10) * sd + mu).reshape(N, N)
V_line = line.predict(grid / 10).reshape(N, N)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.9))
im0 = ax[0].imshow(V_true, origin='lower', cmap='viridis'); ax[0].set_title('True price tags ($)')
ax[0].scatter(seen[:, 1], seen[:, 0], c='white', s=18, label=f'the {SAMPLES} we computed'); ax[0].legend(loc='upper left', fontsize=8)
fig.colorbar(im0, ax=ax[0], shrink=0.8)
for a, M, t in zip(ax[1:], [V_net, V_line], [f'Neural network mistakes (saw {SAMPLES})', f'Straight line mistakes (saw {SAMPLES})']):
    im = a.imshow(M - V_true, origin='lower', cmap='RdBu_r', vmin=-5, vmax=5); a.set_title(t)
fig.colorbar(im, ax=ax[1:], shrink=0.8, label='learned − true ($)')
for a in ax: a.set_xlabel('product 2 on the shelf'); a.set_ylabel('product 1 on the shelf')
plt.show()
print(f'Average miss on all 121 tags:  neural network ${np.abs(V_net - V_true).mean():.2f}   straight line ${np.abs(V_line - V_true).mean():.2f}')

**✅ What you should see:** the true price tags on the left, then two maps of **mistakes** (white = right on, red = too high, blue = too low). The network is pale almost everywhere. The straight line is wrong in a *pattern*: too high in some corners, too low in others. Average miss: the network is off by about **\$0.60** per price tag, the line by about **\$1.50**.

That doesn't sound like a big gap. But price tags are for **deciding**. The real test: run the shop with each set of learned tags.

### Step 4.3 — The real test: run the shop with the learned tags

In [ ]:
# @title ▶️ Run me: what are the learned playbooks worth?
def value_of(playbook):
    # follow a playbook forever and add up the discounted money (same loop as Session 10, no 'best over choices')
    V = np.zeros((N, N))
    for _ in range(5000):
        W = L @ V @ L.T; V_new = np.zeros((N, N))
        for s1 in range(N):
            for s2 in range(N):
                a1, a2 = actions[playbook[s1, s2]]; t1, t2 = s1 + a1, s2 + a2
                V_new[s1, s2] = g[t1] + g[t2] - 8 * (a1 + a2) - (FEE if a1 + a2 > 0 else 0) + GAMMA * W[t1, t2]
        if np.abs(V_new - V).max() < 1e-6: break
        V = V_new
    return V_new

play_net, play_line = scores(V_net).argmax(axis=2), scores(V_line).argmax(axis=2)
card('Run the shop with learned price tags', [
    ('Neural network playbook: same move as the best', f'{(play_net == best_play).mean():.0%} of situations'),
    ('Neural network playbook: money lost per situation', f'${(V_true - value_of(play_net)).mean():.2f}'),
    ('Straight-line playbook: same move as the best', f'{(play_line == best_play).mean():.0%} of situations'),
    ('Straight-line playbook: money lost per situation', f'${(V_true - value_of(play_line)).mean():.2f}')])

**✅ What you should see:** the network's playbook makes the **same move as the best playbook in about 97% of situations** and loses only about **\$0.03**. The straight line matches only about **69%** and loses about **\$20** per situation. A small error in the *shape* of the price tags leads to a very different decision.

**This is the core idea of deep reinforcement learning:** when the table of price tags is too big to compute, learn it with a neural network from the situations you *can* afford to see. (Real deep RL learns from experience, not from an answer key; we used one here so we could grade it.) Warren Powell calls this a **value function approximation**: tonight's VFA box.

### ✏️ Your turn
Set `SAMPLES = 10` in Step 4.2 and re-run 4.2 and 4.3. Then try 50. *(✅ With 10 the network's playbook gets much worse: it loses about **\$10** per situation. With 50 it's back to about **\$0.03**. Data still matters, even for neural networks.)*

---
## 🎯 Takeaways
1. **Foundation models are deep learning,** pretrained so you don't have to train. For small and medium tables they are a superb first try.
2. **Your own networks still matter** when data is big, when the data isn't a plain table, when you need a custom goal, and when the table of price tags explodes.
3. **Accuracy is not the goal; decisions are.** The line was less than \$1 worse per price tag, but about \$20 worse per decision. We build on exactly this idea on **December 2**: *predict, then decide.*